ISMB 2024 Tutorial: Multi-omic data integration for microbiome research using scikit-bio

# Merging abundance with taxonomy
Merging abundace with taxonomy in bioms

Goals
   - [x] Merging abundance with taxonomy in bioms

## Preparation

Install the latest version of scikit-bio if it hasn't been (needed for every Google Colab instance).

In [1]:
from importlib.util import find_spec

In [2]:
if find_spec('skbio') is None:
    !pip install -q scikit-bio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 32.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.2/58.2 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 31.4 MB/s eta 0:00:00


In [3]:
import skbio
skbio.__version__

'0.7.0'

Import common libraries.

In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

If you use Google Colab, and would like to directly mount the shared Google Drive folder containing data files, please uncomment and execute the following code.

In [5]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
# # Specify your remote directory
HOME = '/content/drive/MyDrive/sci-kit-bio_local/Data/emp500'

If you use Google Colab or local Jupyter, and would like to download the data file package to the current directory, please uncomment and execute the following code.

If you use local Jupyter, and have already downloaded and extracted the data file package, please specify its directory.

In [ ]:
# # Specify your local directory
#HOME = '/home/drz/Desktop/Data/emp500'

Check if the directory exists by listing its content.

In [ ]:
!ls $HOME

amplicon  assembly  masspec  README.md	sample.tsv  shotgun


## The Food study

## Shotgun sequencing kraken analysis

This tsv already in github is the result of kraken-bracken analysis (alberto), plus test if every taxon isyeast or no (Elaine script)

In [7]:
## Uncomment for food
HOME = '/content/drive/MyDrive/sci-kit-bio_local/FUNGI'
!ls $HOME/tables

07_ITS_1000reads_grouped_counts.tsv
07_ITS_10reads_grouped_counts.tsv
07_kraken_absolute_isyeast_final.tsv
07_Kraken_grouped_counts.tsv
07_occurrence_eukdetect_earth_microbiomes.tsv
foodKraken.biom
Kraken_grouped_counts.tsv
merged_abundance_taxonomy.csv
reads_earth_microbiomes_k2_B.gsheet
sample.gsheet


In [11]:
from skbio import Table

In [13]:
!pip install -q biom-format
!biom convert -i {HOME}/tables/foodKraken.biom -o {HOME}/tables/foodKrakenHDF5.biom --to-hdf5

In [14]:
table = Table.read(f'{HOME}/tables/foodKrakenHDF5.biom')

In [15]:
table.head()

5 x 5 <class 'biom.table.Table'> with 15 nonzero entries (60% dense)

In [16]:
# Get observation metadata
obs_metadata = table.metadata(axis='observation')
obs_ids = table.ids(axis='observation')

# List to store parsed taxonomy data
parsed_taxonomy_list = []

# Iterate through observation metadata and parse taxonomy
for item in obs_metadata:
    taxonomy_ranks = {}
    if 'taxonomy' in item and isinstance(item['taxonomy'], list):
        # Assuming taxonomy is in the format k__Kingdom; p__Phylum; ...
        for rank_info in item['taxonomy']:
            if '__' in rank_info:
                rank_parts = rank_info.split('__')
                if len(rank_parts) == 2:
                    rank_level = rank_parts[0].strip()
                    rank_name = rank_parts[1].strip()
                    # Map shorthand to full names (optional, adjust as needed)
                    rank_mapping = {'k': 'Kingdom', 'p': 'Phylum', 'c': 'Class', 'o': 'Order', 'f': 'Family', 'g': 'Genus', 's': 'Species'}
                    full_rank_name = rank_mapping.get(rank_level, rank_level)
                    taxonomy_ranks[full_rank_name] = rank_name
            else:
                 # Handle cases where taxonomy is just the name without rank prefix
                 # You might need to adjust this based on your specific data format
                 pass # Or add logic to infer rank if possible

    parsed_taxonomy_list.append(taxonomy_ranks)

# Create a DataFrame from the parsed taxonomy data
taxonomy_df = pd.DataFrame(parsed_taxonomy_list, index=obs_ids)

# Display the head of the taxonomy DataFrame
print("Taxonomy DataFrame:")
display(taxonomy_df.head())

Taxonomy DataFrame:


,Kingdom,Phylum,Class,Order,Family,Genus,Species
318829,,Ascomycota,Sordariomycetes,Magnaporthales,Pyriculariaceae,Pyricularia,oryzae
509241,,Ascomycota,Sordariomycetes,Hypocreales,Nectriaceae,Mariannaea,elegans
1436940,,Ascomycota,Leotiomycetes,Thelebolales,Thelebolaceae,Pseudogymnoascus,sp. BL308
2086344,,Ascomycota,Leotiomycetes,Erysiphales,Erysiphaceae,Podosphaera,cerasi
5059,,Ascomycota,Eurotiomycetes,Eurotiales,Aspergillaceae,Aspergillus,flavus


In [17]:
# Convert the skbio.Table to a pandas DataFrame
table_df = table.to_dataframe()

# Display the head of the DataFrame
print("Table as DataFrame:")
display(table_df.head())

Table as DataFrame:


,DRR000713,DRR000714,DRR205701,DRR205702,DRR205703,ERR11641206,ERR11641208,ERR11641210,ERR11641213,ERR11641215,...,SRR9640347,SRR9640348,SRR9640350,SRR9640351,SRR9640352,SRR9640353,SRR9640354,SRR9640355,SRR9640356,SRR9646966
318829,33884.0,42888.0,0,153.0,1231.0,5822.0,379.0,1814.0,607.0,1990.0,...,14704.0,1405.0,238.0,684.0,962.0,14273.0,4576.0,10099.0,22877.0,886.0
509241,18.0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1436940,56.0,47.0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2086344,18.0,43.0,0,8.0,9.0,211.0,0,595.0,254.0,30407.0,...,20.0,0,0,0,84.0,0,0,35.0,23.0,0
5059,21135.0,25693.0,0,1739.0,11463.0,5769.0,0,2215.0,0,4222.0,...,25817.0,0,0,6507.0,6770.0,31876.0,18494.0,43932.0,69248.0,0


In [18]:
# Merge the abundance table DataFrame with the taxonomy DataFrame
merged_df = table_df.merge(taxonomy_df, left_index=True, right_index=True, how='left')

# Display the head of the merged DataFrame
print("Merged Abundance and Taxonomy DataFrame:")
display(merged_df.head())

Merged Abundance and Taxonomy DataFrame:


,DRR000713,DRR000714,DRR205701,DRR205702,DRR205703,ERR11641206,ERR11641208,ERR11641210,ERR11641213,ERR11641215,...,SRR9640355,SRR9640356,SRR9646966,Kingdom,Phylum,Class,Order,Family,Genus,Species
318829,33884.0,42888.0,0,153.0,1231.0,5822.0,379.0,1814.0,607.0,1990.0,...,10099.0,22877.0,886.0,,Ascomycota,Sordariomycetes,Magnaporthales,Pyriculariaceae,Pyricularia,oryzae
509241,18.0,0,0,0,0,0,0,0,0,0,...,0,0,0,,Ascomycota,Sordariomycetes,Hypocreales,Nectriaceae,Mariannaea,elegans
1436940,56.0,47.0,0,0,0,0,0,0,0,0,...,0,0,0,,Ascomycota,Leotiomycetes,Thelebolales,Thelebolaceae,Pseudogymnoascus,sp. BL308
2086344,18.0,43.0,0,8.0,9.0,211.0,0,595.0,254.0,30407.0,...,35.0,23.0,0,,Ascomycota,Leotiomycetes,Erysiphales,Erysiphaceae,Podosphaera,cerasi
5059,21135.0,25693.0,0,1739.0,11463.0,5769.0,0,2215.0,0,4222.0,...,43932.0,69248.0,0,,Ascomycota,Eurotiomycetes,Eurotiales,Aspergillaceae,Aspergillus,flavus


In [19]:
# Define the sample ID
sample_id = 'DRR000713'

# Calculate the total number of reads for the sample from table_df
total_reads_sample = table_df[sample_id].sum()

print(f"Total number of reads for sample {sample_id} from table_df: {total_reads_sample}")

# To get reads per phylum from table_df, we need to merge with taxonomy information
# We can use the taxonomy_df created in cell 6dnHIS40DT-N
table_with_taxonomy = table_df.merge(taxonomy_df, left_index=True, right_index=True, how='left')

# Calculate the total number of reads per phylum for the sample
# Group by 'Phylum' and sum the abundance for the specific sample
phylum_abundance_sample = table_with_taxonomy.groupby('Phylum')[sample_id].sum()

print(f"\nTotal number of reads per phylum for sample {sample_id} from table_df:")
display(phylum_abundance_sample)

# Calculate and print the sum of reads per phylum
sum_of_phylum_reads = phylum_abundance_sample.sum()
print(f"\nSum of reads per phylum for sample {sample_id} from table_df: {sum_of_phylum_reads}")

Total number of reads for sample DRR000713 from table_df: 118046.0

Total number of reads per phylum for sample DRR000713 from table_df:


,DRR000713
Phylum,
,5279.0
Ascomycota,112250.0
Basidiomycota,517.0
Blastocladiomycota,0
Chytridiomycota,0
Cryptomycota,0
Microsporidia,0
Mucoromycota,0
Zoopagomycota,0



Sum of reads per phylum for sample DRR000713 from table_df: 118046.0


In [20]:
# Define the output file path
output_file_path = f'{HOME}/tables/FOOD_merged_abundance_taxonomy.csv'

# Print the merged_df DataFrame to a CSV file
merged_df.to_csv(output_file_path)

print(f"DataFrame successfully saved to {output_file_path}")

DataFrame successfully saved to /content/drive/MyDrive/sci-kit-bio_local/FUNGI/tables/FOOD_merged_abundance_taxonomy.csv
